# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of analysis

One row represents the daily search performance of a single content item for a single client on a specific report date.

## Time window

The warehouse data covers report dates from **2025-01-27** to **2026-06-30**. Features for the model will be created using historical data before the prediction point, while future data will only be used to create labels and evaluate the model. This separation helps prevent data leakage.

In [1]:
import sys
print(sys.executable)

c:\Users\adity\OneDrive\Documents\FlyRank ML Intership\flyrank-ml-internship-starter\.venv\Scripts\python.exe


In [2]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token: ")

In [3]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [4]:
# Verify a mid-panel month (recommended for development)

con.sql("""
SELECT
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(*) AS total_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

,first_date,last_date,total_rows
0,2026-03-01,2026-03-31,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
These fields are used as input for analysis or model building because they are observable measurements collected before the prediction point.

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engagement_rate

### Label
The target will be a future content opportunity or performance outcome created from a later time window. Future observations are used only as labels to avoid data leakage.

### Context
These fields provide identification and grouping but are not predictive features.

- client_hash_id
- content_hash_id
- report_date

### Excluded
The following fields are excluded because they may introduce leakage or are only used for joining tables.

- Join keys (client_hash_id, content_hash_id)
- Any future-window metrics
- Product-generated scores or decision outputs (if available)

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Verification Queries

The following queries verify the data contract:

1. Check the dataset grain.
2. Verify the number of rows and the report date range for a development month.
3. Verify data availability using the availability flag.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [9]:
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS records
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""").df()

,report_date,client_hash_id,content_hash_id,records


In [10]:
con.sql("""
DESCRIBE
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [11]:
con.sql("""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE ga4_data_available IS TRUE
""").df()

,available_rows
0,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

This dataset has the following limitations:

- Different clients have different amounts of historical data, creating an unbalanced panel.
- Some early records contain only Google Search Console data because GA4 tracking was not yet available.
- The dataset contains observable signals only and does not include product-generated decision scores.
- This analysis cannot prove that refreshing content causes better performance; it can only identify patterns and opportunities.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check the availability of GA4 data

con.sql(f"""
SELECT
    ga4_data_available,
    COUNT(*) AS rows
FROM {TABLES['fact_daily']}
GROUP BY ga4_data_available
ORDER BY ga4_data_available
""").df()

,ga4_data_available,rows
0,False,46383873
1,True,2816455
2,<NA>,29635327


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.